# Task 1.2: From Notebook to Production — Image Caption Generation
## Notebook 01: Exploratory Data Analysis & Preprocessing Pipeline

This notebook covers:
1. **Dataset Ingestion**: Loading and exploring the **Flickr8k Dataset** (8,091 images $\times$ 5 paired captions).
2. **Caption Text Analysis**: Token length distributions, vocabulary frequency, and vocabulary pruning.
3. **Image Exploration**: Resolution inspection, normalization stats, and torchvision transformations.
4. **Vocabulary Construction**: Adding special tokens (`<pad>`, `<start>`, `<end>`, `<unk>`) and JSON serialization.
5. **Data Leakage-Free Splitting**: Splitting by unique image IDs into Train (80%), Val (10%), and Test (10%).

In [ ]:
import os
import re
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from collections import Counter

# Import modular components from src package
import sys
sys.path.append("..")
from src.config import get_default_config
from src.data.vocabulary import Vocabulary
from src.data.dataset import parse_flickr8k_captions, split_flickr8k_data
from src.data.transforms import get_train_transforms, get_val_transforms

cfg = get_default_config()
print(f"Captions File: {cfg.paths.captions_file} (Exists: {cfg.paths.captions_file.exists()})")
print(f"Images Directory: {cfg.paths.images_dir} (Exists: {cfg.paths.images_dir.exists()})")

### 1. Ingesting and Inspecting Captions

In [ ]:
df = parse_flickr8k_captions(cfg.paths.captions_file)
print(f"Total Caption Records: {len(df):,}")
print(f"Total Unique Images: {df['image_name'].nunique():,}")
df.head(10)

### 2. Caption Length & Vocabulary Frequency Distributions

In [ ]:
# Tokenize and analyze caption lengths
df['cleaned_caption'] = df['caption'].apply(Vocabulary.clean_caption)
df['token_count'] = df['cleaned_caption'].apply(lambda c: len(c.split()))

print(df['token_count'].describe())

# Plot caption length histogram
plt.figure(figsize=(10, 4))
plt.hist(df['token_count'], bins=30, color='#1e3c72', edgecolor='black', alpha=0.8)
plt.title('Caption Word Count Distribution (Flickr8k)', fontsize=14, fontweight='bold')
plt.xlabel('Number of Words', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

### 3. Word Frequencies & Thresholding

In [ ]:
all_words = [w for cap in df['cleaned_caption'] for w in cap.split()]
word_counts = Counter(all_words)
print(f"Total words in corpus: {len(all_words):,}")
print(f"Unique vocabulary count: {len(word_counts):,}")

top_20 = word_counts.most_common(20)
words, freqs = zip(*top_20)

plt.figure(figsize=(12, 5))
plt.bar(words, freqs, color='#2a5298')
plt.title('Top 20 Most Frequent Words in Flickr8k', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.ylabel('Frequency')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

### 4. Splitting Data (Zero Leakage) & Building Vocabulary

In [ ]:
train_df, val_df, test_df = split_flickr8k_data(df, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1, seed=42)
print(f"Train Samples: {len(train_df):,} ({train_df['image_name'].nunique()} images)")
print(f"Val Samples:   {len(val_df):,} ({val_df['image_name'].nunique()} images)")
print(f"Test Samples:  {len(test_df):,} ({test_df['image_name'].nunique()} images)")

# Build Vocabulary strictly from training set to avoid data leakage
vocab = Vocabulary(min_freq=3)
vocab.build_vocabulary(train_df['caption'].tolist())
vocab.save(cfg.paths.vocab_path)

print(f"Vocabulary built with {len(vocab):,} tokens.")
print(f"Special Tokens: <pad>={vocab.pad_idx}, <start>={vocab.start_idx}, <end>={vocab.end_idx}, <unk>={vocab.unk_idx}")

### 5. Inspecting Sample Images and Reference Captions

In [ ]:
sample_row = train_df.iloc[0]
img_name = sample_row['image_name']
img_path = cfg.paths.images_dir / img_name

if img_path.exists():
    img = Image.open(img_path)
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Image: {img_name}", fontsize=12, fontweight='bold')
    plt.show()

    captions = df[df['image_name'] == img_name]['caption'].tolist()
    print("5 Human Reference Captions:")
    for i, c in enumerate(captions, 1):
        print(f"{i}. {c}")